In [14]:
import pandas as pd
import numpy as np
import re

In [15]:
df_asort = pd.read_csv("data/raw/Asort_2026-04-25.csv", sep=',', on_bad_lines='skip')
df_pozd = pd.read_parquet("data/raw/POZD.parquet")
df_dok = pd.read_parquet("data/raw/DOK.parquet")
df_tow = pd.read_parquet("data/raw/TOWAR.parquet")

In [ ]:
# # plik Towar z duplikatami SKU
# df_tow_pre = pd.read_parquet("data/interim/Towar_prefab_full.parquet")

In [16]:
polish_map = str.maketrans(
    "ąćęłńóśźżĄĆĘŁŃÓŚŹŻ",
    "acelnoszzACELNOSZZ"
)

def clean_name(text: str) -> str:
    if pd.isna(text):
        return ""

    # lower
    text = text.lower()

    # ręczna zamiana polskich znaków 
    text = text.translate(polish_map)

    # zostaw tylko litery/cyfry/spacje
    text = re.sub(r'[^a-z0-9 ]', ' ', text)

    # redukcja wielokrotnych spacji
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


In [17]:
# ====================================================
# 1. PRAWIDŁOWA NORMALIZACJA POLSKICH ZNAKÓW (bez zjadania liter!)
# ====================================================

polish_map = str.maketrans({
    "ą": "a", "ć": "c", "ę": "e", "ł": "l",
    "ń": "n", "ó": "o", "ś": "s", "ź": "z", "ż": "z",
    "Ą": "A", "Ć": "C", "Ę": "E", "Ł": "L",
    "Ń": "N", "Ó": "O", "Ś": "S", "Ź": "Z", "Ż": "Z"
})

def clean_name(text: str):
    # if pd.isna(text):
    #     return ""
    text = str(text)
    text = text.translate(polish_map)         # zamiana PL → ASCII 1:1
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)   # zamiana reszty na spację
    text = re.sub(r"\s+", " ", text).strip()  # redukcja wielokrotnych spacji
    return text

def process_df(df):
    df = df.copy()

    # dodanie kolumny clean_name
    df["clean_name"] = df["Nazwa"].astype(str).apply(clean_name)

    return df

# ====================================================
# UŻYCIE
# ====================================================
# df_result = process_df(df_xxx)


In [18]:
print(clean_name("MASŁO EXTRA KOST 200G WŁOSZCZOWA"))
# maslo extra kost 200g wloszczowa
print(clean_name("ZIAŁOŻÓŹĆĄĘŃŁ"))
# zialozozcaenl
print(clean_name("BIBLIOTEKA ZDROWIA – KURACJE OCZYSZCZAJĄCE!"))
# biblioteka zdrowia kuracje oczyszczajace
print(clean_name("200 g 450 1,5kg 2.4kg '."))


maslo extra kost 200g wloszczowa
zialozozcaenl
biblioteka zdrowia kuracje oczyszczajace
200 g 450 1 5kg 2 4kg


In [ ]:
# df_tow_clean = process_df(df_tow)

In [20]:
df_pozd['TypTowaru'].value_counts()

0    4256616
1      11768
3       2356
6       1698
2        218
Name: TypTowaru, dtype: int64

In [43]:
df_tow['Aktywny'].value_counts()

0    21524
1    19854
Name: Aktywny, dtype: int64

In [62]:
towar_pre = df_tow_pre[['TowId', 'TowId_Master', 'Nazwa', 'Aktywny']] # , 'clean_name', 'AsId'
towar_pre

,TowId,TowId_Master,Nazwa,Aktywny
0,12851,12851,GREENERS MASŁO IRLANDZKIE TRADYCYJNE 200G,0
1,18155,18163,MARG KUB RAMA Z MASŁEM 225G UNILEVER,0
2,18156,18156,MARG KUB RAMA MASŁO Z SOLĄ 225G UNILEVER,0
3,18163,18163,MARG KUB RAMA Z MASŁEM 225G UNILEVER,0
4,11803,11803,LYZECZKI PLAST.JEDN12 SZT,0
...,...,...,...,...
2824,52426,53196,MARG FLORA PRO-ACTIV 225G UPIELD,0
2825,55334,55449,MARG FLORA 225G UPFIELD,0
2826,66194,66488,MIX FINUU HYVAA KOSTKA 200G KRUSZWICA,1
2827,66902,66902,MIKS ELEPLANT 79% 200G KRUSZWICA,1


In [63]:
towar = df_tow[['TowId', 'Nazwa', 'AsId', 'JMId']]
towar

,TowId,Nazwa,AsId,JMId
0,19897,DOMOWE PRZETWORY I NALEWKI,273,1
1,19898,RYSUJE SZLACZKI ZANIM NAUCZE SIE PISAC 1,273,1
2,19899,NIELEGALNI KOLEKCJA /KSIAZ/ 2,273,1
3,19900,KOLEKCJA OKRETOW WOJENNYCH 10,273,1
4,19906,ABC W SWIECIE PRZEDSZKOLAKA 1,273,1
...,...,...,...,...
41373,78266,CHRUPKI CURLY SŁ KARMEL 100G LORENZ,166,1
41374,78268,SOS HOD DOG&WRAP 440G DEVELEY,323,1
41375,78366,"WODA KUBUŚ ACTIVE CYTRYNA 0,75L MASPEX",99,1
41376,78471,"NAPÓJ KUBUŚ WATERRR CZER OWOCE 0,75L TYMBARK",99,1


In [64]:
aktywny = df_tow[df_tow['Aktywny']==1][['TowId', 'Nazwa', 'AsId', 'JMId']]
aktywny

,TowId,Nazwa,AsId,JMId
1568,54929,"WINO VALPILCELLA RIPASSO SUPERIORE 0,75L C/W",353,1
1569,54932,WINO PROSECCO SU P DOCG TRAMIOL 750ML,353,1
1570,54933,WINO LOCOROTONDO MESSAPICO 750ML (,353,1
1571,54936,WINO BARBERA D'A STI DOCG PRATOMAN,353,1
1601,19445,100zł VECTONE;VEC;294; 100,47,1
...,...,...,...,...
41368,77964,CAPUCINO TARTY MALINOWA 40G MOKATE,106,1
41369,78156,POMYSŁ NA KURCZAK 5 SMAKÓW 37G NESTLE,37,1
41370,78157,POMYSŁ NA MAKARON TAJSKI 26G NESTLE,37,1
41371,78159,FIX KNORR MAC'N CHEESE 33G UNILEVER,37,1


In [65]:
counts = towar_pre.groupby('TowId_Master')['TowId'].count()
duplikaty = counts[counts > 2]
duplikaty

TowId_Master
15894    3
18310    3
24474    3
25747    3
36371    3
36929    3
37889    3
39025    4
51115    8
Name: TowId, dtype: int64

In [66]:
mask = towar_pre.groupby('TowId_Master')['TowId'].transform('count') > 2
wynik = towar_pre[mask].sort_values('TowId_Master')
wynik

,TowId,TowId_Master,Nazwa,Aktywny
79,15892,15894,MASŁO OSEŁKA GÓRSKA SMAKOWA MIX 80G SOBIK,0
80,15893,15894,MASŁO OSEŁKA GÓRSKA SMAKOWA MIX 80G SOBIK,0
81,15894,15894,MASŁO OSEŁKA GÓRSKA SMAKOWA MIX 80G SOBIK,0
39,4337,18310,MARG KUB RAMA CLASSIC 450G UNILEVER,0
82,18044,18310,MARG KUB RAMA CLASSIC 450G UNILEVER,0
1554,18310,18310,MARG KUB RAMA CLASSIC 450G UNILEVER,0
95,24472,24474,PRZECENA MASŁO EXTRA KOST Z KOŃSKICH 200G KOŃSKIE,0
1564,6956,24474,MASŁO EXTRA KOST Z KOŃSKICH 200G KOŃSKIE,1
1570,24474,24474,PRZECENA MASŁO EXTRA KOST Z KOŃSKICH 200G KOŃSKIE,0
2253,25747,25747,PRZECENA JOGURT FANTASIA WEDEL KREM KARMELLOVE...,1


In [52]:
nie_aktywny = df_tow[df_tow['Aktywny']==0][['TowId', 'Nazwa', 'AsId', 'JMId']]
nie_aktywny

,TowId,Nazwa,AsId,JMId
0,19897,DOMOWE PRZETWORY I NALEWKI,273,1
1,19898,RYSUJE SZLACZKI ZANIM NAUCZE SIE PISAC 1,273,1
2,19899,NIELEGALNI KOLEKCJA /KSIAZ/ 2,273,1
3,19900,KOLEKCJA OKRETOW WOJENNYCH 10,273,1
4,19906,ABC W SWIECIE PRZEDSZKOLAKA 1,273,1
...,...,...,...,...
41373,78266,CHRUPKI CURLY SŁ KARMEL 100G LORENZ,166,1
41374,78268,SOS HOD DOG&WRAP 440G DEVELEY,323,1
41375,78366,"WODA KUBUŚ ACTIVE CYTRYNA 0,75L MASPEX",99,1
41376,78471,"NAPÓJ KUBUŚ WATERRR CZER OWOCE 0,75L TYMBARK",99,1


In [35]:
df_pozd.columns

Index(['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus',
       'IloscMinus', 'PoziomCen', 'Metoda', 'CenaDomyslna', 'CenaPrzedRab',
       'RabatProc', 'CenaPoRab', 'Wartosc', 'CenaDet', 'CenaMag', 'Stawka',
       'TypTowaru', 'IleWZgrzewce', 'StawkaDod', 'Netto', 'Podatek',
       'SledzPartii', 'UUID'],
      dtype='object')

In [ ]:
# df_tow_clean
tow_cols = ['TowId', 'AsId', 'JMId', 'Nazwa', 'Kod', 'Opis1', 'Producent', 'Marza', 'Stawka', 'Aktywny']
df_tow_selected = df_tow[tow_cols].copy()

# df_asort
asort_cols = ['AsId', 'Nazwa']
df_asort_selected = df_asort[asort_cols].copy()
df_asort_selected = df_asort_selected.rename(columns={'Nazwa': 'Nazwa_asort'})

# df_dok
dok_cols = ['DokId', 'Data', 'KolejnyWDniu', 'NrDok', 'TypDok', 'Aktywny', 'Razem', 'DoZaplaty', 'Zaplacono']
df_dok_selected = df_dok[dok_cols].copy()

# df_pozd
pozd_cols = ['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPrzedRab', 'CenaPoRab', 'Wartosc', 'CenaDet']
df_pozd_selected = df_pozd[pozd_cols].copy()

print(f"df_tow_selected:   {df_tow_selected.shape}")
print(f"df_asort_selected: {df_asort_selected.shape}")
print(f"df_dok_selected:   {df_dok_selected.shape}")
print(f"df_pozd_selected:   {df_pozd_selected.shape}")

df_tow_selected:   (41378, 10)
df_asort_selected: (337, 2)
df_dok_selected:   (952122, 9)
df_pozd_selected:   (4272656, 11)


In [37]:
df_tow_selected.to_parquet("data/interim/Towar-columns_selected-records_full.parquet", compression='zstd', index=False)
df_asort_selected.to_parquet("data/interim/Asort-columns_selected-records_full.parquet", compression='zstd', index=False)
df_dok_selected.to_parquet("data/interim/Dok-columns_selected-records_full.parquet", compression='zstd', index=False)
df_pozd_selected.to_parquet("data/interim/PozDok-columns_selected-records_full.parquet", compression='zstd', index=False)

# parametry dla kompresji przy zapisie .parquet
# snappy  — najszybszy odczyt, słabsza kompresja - domyślny 
# gzip    — dobry balans
# zstd    — bardzo dobry balans (szybkość + kompresja)
# brotli  — najmniejszy plik, wolniejszy odczyt